# Ticket Triage: measure the original model first

**Purpose:** find out how well the original Qwen3-0.6B performs before training. This notebook does not fine-tune a model, publish anything, or call a paid inference API.

Run each cell in order using its play button. All model downloads and predictions happen on Colab's machine. Public ungated Qwen downloads normally need no Hugging Face token. Free GPU availability is not guaranteed.

**Before opening this notebook:** run `python3 scripts/prepare_colab.py` in the local project. It creates `artifacts/colab-project.zip`. You will upload that ZIP below. Upload this `.ipynb` through Colab's Open notebook → Upload dialog.

Select **Runtime → Change runtime type → GPU** (for example T4, if offered), then connect. If no free GPU is available, stop and try again later; you do not need a paid plan for these files.


## 1. Check the GPU

This confirms that Colab—not your laptop—has a GPU ready. Stop here if the check fails.

In [1]:
import sys
import torch
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
assert torch.cuda.is_available(), "No GPU connected. Select a GPU runtime, reconnect, then rerun."
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))


Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
PyTorch: 2.14.1+cu130
GPU: Tesla T4
GPU memory (GiB): 14.6


## 2. Upload the project ZIP

Choose `colab-project.zip`, not the notebook itself. This package contains code and datasets, without your secrets or model weights. Extraction below accepts only regular files inside the project folder.

In [2]:
from google.colab import files
from pathlib import Path
from zipfile import ZipFile
import io
import os
import shutil

uploaded = files.upload()
assert len(uploaded) == 1, "Upload exactly one colab-project.zip file."
zip_bytes = next(iter(uploaded.values()))
project = Path("/content/ticket-triage")
assert not project.exists(), "Project already exists. Skip this cell if already uploaded, or use a fresh runtime."
staging = Path("/content/triage-upload")
staging.mkdir(exist_ok=True)
with ZipFile(io.BytesIO(zip_bytes)) as archive:
    for member in archive.infolist():
        relative = Path(member.filename)
        if relative.is_absolute() or ".." in relative.parts or not relative.parts or relative.parts[0] != "ticket-triage":
            raise ValueError("Unexpected path in ZIP")
        if (member.external_attr >> 16) & 0o170000 == 0o120000:
            raise ValueError("Symbolic links are not accepted")
        if member.is_dir():
            continue
        target = staging / relative
        target.parent.mkdir(parents=True, exist_ok=True)
        with archive.open(member) as source, target.open("wb") as destination:
            shutil.copyfileobj(source, destination)
(staging / "ticket-triage").rename(project)
os.chdir(project)
print("Working folder:", project)
del uploaded, zip_bytes


Saving colab-project.zip to colab-project.zip
Working folder: /content/ticket-triage


## 3. Install the baseline library

Use Colab’s installed PyTorch/CUDA build. Transformers is pinned for this experiment; all actual package versions are saved in the downloaded results. This stack still needs its first Colab smoke run. If Colab explicitly asks for a runtime restart, restart the session and continue from this cell, then the next cells; do not upload again.

In [6]:
import os
from pathlib import Path
os.chdir("/content/ticket-triage")
%pip install -q -r training/requirements-baseline.txt
import subprocess
import sys

def run(*arguments):
    subprocess.run([sys.executable, *map(str, arguments)], check=True)

run("-m", "evaluation.validate_data")


In [7]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-m", "evaluation.validate_data"],
    cwd="/content/ticket-triage",
    capture_output=True,
    text=True,
)

print("Exit code:", result.returncode)
print(result.stdout)
print(result.stderr)

Exit code: 0
{
  "counts": {
    "train": 128,
    "validation": 34,
    "test": 66
  },
  "errors": []
}




## 4. Check token lengths — tokenizer only

A token is a small piece of text. The full prompt, ticket, and correct answer must fit inside the training budget. This cell downloads only the tokenizer, checks training/validation data, and never truncates examples. The test set stays unused. A 512-token budget is a proposal, not an established requirement.

In [8]:
import json
from datetime import datetime, timezone
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
ROOT = Path("/content/ticket-triage")
TOKEN_DIR = ROOT / "artifacts" / (RUN_ID + "-token-check")
run("-m", "evaluation.run_model", "--input", "data/validation.jsonl",
    "--check-data", "data/train.jsonl", "--check-only", "--output-dir", TOKEN_DIR)
metadata = json.loads((TOKEN_DIR / "metadata.json").read_text())
MODEL_REVISION = metadata["resolved_revision"]
lengths = json.loads((TOKEN_DIR / "token_lengths.json").read_text())
for name, report in lengths.items():
    print(name, "longest full example:", report["max_full_training_tokens"],
          "examples over 512:", len(report["over_budget_ids"]))
print("Fixed model commit for this experiment:", MODEL_REVISION)
print("If examples exceed 512, keep the report. We will choose the training budget before fine-tuning.")


data/validation.jsonl longest full example: 313 examples over 512: 0
data/train.jsonl longest full example: 317 examples over 512: 0
Fixed model commit for this experiment: c1899de289a04d12100db370d81485cdf75e47ca
If examples exceed 512, keep the report. We will choose the training budget before fine-tuning.


## 5. Try two tickets first

This is a smoke check: does the model load and answer? It downloads the base weights into Colab. The saved commit ensures this run uses the same model as the tokenizer check. It does not train anything. The model sees ticket text only, never the expected answer. Raw invalid responses are preserved rather than repaired.

In [9]:
from evaluation.validate_data import read_jsonl
rows = read_jsonl("data/validation.jsonl")
smoke_input = ROOT / "artifacts" / (RUN_ID + "-smoke-input.jsonl")
smoke_input.write_text("".join(json.dumps(row) + "\n" for row in rows[:2]))
SMOKE_DIR = ROOT / "artifacts" / (RUN_ID + "-smoke")
run("-m", "evaluation.run_model", "--input", smoke_input,
    "--revision", MODEL_REVISION, "--device", "cuda", "--output-dir", SMOKE_DIR)
for row in read_jsonl(SMOKE_DIR / "predictions.jsonl"):
    print(row["id"], row["prediction"])
    if row["hit_output_limit"]:
        print("Output hit its token limit; inspect before the full run.")


val-001 {
  "category": "other",
  "environment": "staging",
  "customer_reported_impact": "The password reset link for staging has already been used.",
  "suggested_team": "billing_support"
}
val-002 {
  "category": "rate_limit",
  "environment": "production",
  "customer_reported_impact": "429 responses during request bursts delaying message processing",
  "suggested_team": "billing_support"
}


## 6. Run all validation tickets and calculate starting scores

If the two-ticket check ran, run this cell. It makes one prediction per validation ticket, then calculates scores. Greedy decoding is a deliberate repeatable baseline choice for this short JSON task; it differs from Qwen’s suggested sampling recipe. No output score is promised. Do not change the test set based on these results.

### Your first Step 6 output

You reported these scores from the original model on 34 validation tickets:

```text
schema_validity_rate: 0.500
category_macro_f1: 0.364
environment_accuracy: 0.441
team_accuracy: 0.118
Team follows category; it is not an independent capability.
Impact summaries require human review; these scores do not measure semantic summary quality.
```

These are your reported baseline results, not expected outputs for every future run. The downloaded `metrics.json` is the source for exact, unrounded values.

### What each score means

| Score | Simple meaning | Your reported result |
| --- | --- | --- |
| `schema_validity_rate` | Did the answer follow the required JSON structure, with exactly the required fields, allowed values, and valid types? Correct structure does not guarantee correct meaning. | 17 out of 34 responses passed: 50.0%. |
| `category_macro_f1` | Calculate an F1 score for each of the six ticket categories, then average those six scores equally. | 0.364; this does not mean 36.4% of tickets were correct. |
| `environment_accuracy` | Did the model correctly identify production, staging, development, or unknown in a schema-valid response? | 15 out of 34 responses counted as correct: 44.1%. |
| `team_accuracy` | Did the model select the correct support team in a schema-valid response? | 4 out of 34 responses counted as correct: 11.8%. |

**Our scoring rule:** an invalid-schema response counts as incorrect for all task metrics, even if an individual field happens to be right. For category F1, it counts as a missed ticket for the true category. These scores therefore include format failures as well as classification errors.

The team is determined by category: billing → billing_support; other → general_support; all remaining categories → developer_support. Team accuracy is not an independent reasoning capability. Impact summary factuality and completeness require human review; none of these four scores measures semantic summary quality.

### Category macro-F1: simple calculation example

There is no **k** here. Precision@k and Recall@k apply to ranked lists. Our model predicts one category per ticket, so we use ordinary classification precision and recall.

Consider billing. **These counts are illustrative, not your actual prediction counts:**

- 5 tickets actually belong to billing.
- The model calls 4 tickets billing.
- 3 of those billing predictions are correct.

**Precision: “Of the tickets the model called billing, how many really were billing?”**

```text
Precision = correct billing predictions ÷ all billing predictions
          = 3 ÷ 4
          = 0.75, or 75%
```

**Recall: “Of all the actual billing tickets, how many did the model find?”**

```text
Recall = correct billing predictions ÷ actual billing tickets
       = 3 ÷ 5
       = 0.60, or 60%
```

Then billing’s F1 combines those:

```text
F1 = 2 × precision × recall ÷ (precision + recall)
   = 2 × 0.75 × 0.60 ÷ (0.75 + 0.60)
   = 0.667 (rounded, for this example)
```

Equivalently, this example has 3 correct predictions (TP), 1 false billing prediction (FP), and 2 missed billing tickets (FN):

```text
F1 = 2 × TP ÷ (2 × TP + FP + FN)
   = 6 ÷ (6 + 1 + 2)
   = 0.667 (rounded)
```

Calculate F1 separately for authentication, billing, rate_limit, integration, bug, and other, then average:

```text
Macro-F1 = (authentication F1 + billing F1 + rate_limit F1
            + integration F1 + bug F1 + other F1) ÷ 6
```

Your actual six category F1 values are stored in `metrics.json` under `category_per_class`. Their unrounded average produced the reported **0.364**. We cannot reconstruct those individual values from the rounded overall score alone. Higher F1 is better; 1.0 is perfect. In this evaluator a category with no correct predictions receives F1 = 0.



In [10]:
BASELINE_DIR = ROOT / "artifacts" / (RUN_ID + "-base-validation")
run("-m", "evaluation.run_model", "--input", "data/validation.jsonl",
    "--revision", MODEL_REVISION, "--device", "cuda", "--output-dir", BASELINE_DIR)
run("-m", "evaluation.evaluate", "--gold", "data/validation.jsonl",
    "--predictions", BASELINE_DIR / "predictions.jsonl",
    "--output", BASELINE_DIR / "metrics.json")
metrics = json.loads((BASELINE_DIR / "metrics.json").read_text())
for key in ("schema_validity_rate", "category_macro_f1", "environment_accuracy", "team_accuracy"):
    print(f"{key}: {metrics[key]:.3f}")
print("Team follows category; it is not an independent capability.")
print("Impact summaries require human review; these scores do not measure semantic summary quality.")


schema_validity_rate: 0.500
category_macro_f1: 0.364
environment_accuracy: 0.441
team_accuracy: 0.118
Team follows category; it is not an independent capability.
Impact summaries require human review; these scores do not measure semantic summary quality.


## 7. Inspect the model’s mistakes

Look at real ticket text, expected answers, and raw predictions. Mark incorrect categories, missing facts, invented facts, and malformed JSON. These are validation cases: use them to understand where training may help.

In [11]:
predictions = {r["id"]: r for r in read_jsonl(BASELINE_DIR / "predictions.jsonl")}
from evaluation.contract import output_errors
for row in read_jsonl("data/validation.jsonl"):
    prediction = predictions[row["id"]]
    try:
        parsed = json.loads(prediction["prediction"])
    except json.JSONDecodeError:
        parsed = None
    if output_errors(parsed) or parsed != row["expected"]:
        print("\nID:", row["id"], "\nTICKET:", row["ticket"])
        print("EXPECTED:", json.dumps(row["expected"]))
        print("MODEL:", prediction["prediction"])
print("Different wording in an impact summary is not necessarily a mistake; review its meaning.")



ID: val-001 
TICKET: The password reset link for staging says it has already been used, so I cannot reset my password.
EXPECTED: {"category": "authentication", "environment": "staging", "customer_reported_impact": "The customer cannot reset the staging password.", "suggested_team": "developer_support"}
MODEL: {
  "category": "other",
  "environment": "staging",
  "customer_reported_impact": "The password reset link for staging has already been used.",
  "suggested_team": "billing_support"
}

ID: val-002 
TICKET: Our production worker receives 429 responses during request bursts, delaying message processing.
EXPECTED: {"category": "rate_limit", "environment": "production", "customer_reported_impact": "Message processing is delayed during request bursts.", "suggested_team": "developer_support"}
MODEL: {
  "category": "rate_limit",
  "environment": "production",
  "customer_reported_impact": "429 responses during request bursts delaying message processing",
  "suggested_team": "billing_s

## 8. Download your results before disconnecting

Colab storage is temporary. Download this ZIP and keep it in your local project’s `artifacts/` folder. It contains raw predictions, scores, token reports, run metadata, and installed package versions. It contains no model weights or credentials. Rerunning a completed inference cell requires a new run name; the runner refuses to overwrite evidence.

In [12]:
import importlib.metadata
package_versions = sorted(f"{dist.metadata['Name']}=={dist.version}"
                          for dist in importlib.metadata.distributions() if dist.metadata.get("Name"))
(ROOT / "artifacts" / "colab-packages.txt").write_text("\n".join(package_versions) + "\n")
archive_path = shutil.make_archive("/content/triage-baseline-results", "zip", ROOT / "artifacts")
files.download(archive_path)
print("Done. Keep the results ZIP. Next: inspect the baseline and prepare the fine-tuning notebook.")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Done. Keep the results ZIP. Next: inspect the baseline and prepare the fine-tuning notebook.


## What this notebook has and has not completed

Completed when you run it successfully: GPU check, dataset checks, tokenizer length report, baseline predictions, starting scores, and a downloadable result bundle.

Not included yet: LoRA training, adapter evaluation, React, FastAPI, or model publishing. We will prepare training after inspecting the baseline.

References: [Qwen3-0.6B model card](https://huggingface.co/Qwen/Qwen3-0.6B), [Transformers generation](https://huggingface.co/docs/transformers/main_classes/text_generation), [Colab FAQ](https://research.google.com/colaboratory/faq.html).
